# Oracle 26ai Banking Nudge Master Notebook

This notebook is the single source of truth for the proactive banking nudge demo. It follows the same run order as the `26ai-banking-demo` repo while keeping the walkthrough in one place.

Best practice: keep reusable production SQL and helper code in the repo scripts, and use the notebook as the guided entry point that orchestrates those scripts, shows the outputs, and explains the flow. For this demo, the notebook is the main artifact you open; the SQL files remain the canonical implementation.

## What This Notebook Covers

- the real 26ai-banking-demo schema and run order
- in-database ONNX embeddings with `VECTOR_EMBEDDING()`
- vector search with `VECTOR_DISTANCE()`
- property graph traversal with SQL/PGQ
- Select AI and MCP integration
- APEX-facing PL/SQL entry point
- the three demo use cases: card page view, abandoned application, and declined transaction

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
from datetime import datetime, timedelta, timezone

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'oracledb', 'pandas', 'numpy', 'python-dotenv'], check=False)

import pandas as pd
import oracledb
from dotenv import load_dotenv

load_dotenv()

ORACLE_USER = os.getenv('ORACLE_USER', 'testuser')
ORACLE_PASSWORD = os.getenv('ORACLE_PASSWORD', 'TestPass123')
ORACLE_DSN = os.getenv('ORACLE_DSN', 'localhost:1521/FREEPDB1')
MODEL_NAME = os.getenv('ORACLE_MODEL_NAME', 'MINILM_EMB')
ONNX_FILE = os.getenv('ORACLE_ONNX_FILE', 'all_MiniLM_L12_v2.onnx')
DIRECTORY_NAME = os.getenv('ORACLE_DIRECTORY_NAME', 'ONNX_DIR')
DEMO_ROOT = os.getenv('BANKING_DEMO_ROOT', '/workspaces/oracle-26ai-learning/26ai-banking-demo')
SQL_ROOT = os.path.join(DEMO_ROOT, 'sql')

conn = oracledb.connect(user=ORACLE_USER, password=ORACLE_PASSWORD, dsn=ORACLE_DSN)

def run_sql(sql_text, params=None, fetch=False):
    with conn.cursor() as cur:
        if params is None:
            cur.execute(sql_text)
        else:
            cur.execute(sql_text, params)
        if fetch:
            columns = [col[0] for col in cur.description]
            return pd.DataFrame(cur.fetchall(), columns=columns)
    conn.commit()

print('Connected to Oracle AI Database', conn.version)
print('Demo root:', DEMO_ROOT)
print('SQL root:', SQL_ROOT)

## Demo Model And Run Order

The notebook now mirrors the real repository flow rather than a toy schema.

```text
01_schema -> 02_staging_ddl -> 03_load_onnx_model -> 04_copy_data -> 05_transform -> 06_embed_and_index -> 07_property_graph -> 08_select_ai_profile -> 09_uc1 -> 10_uc2 -> 11_uc3
```

Run order from the notebook:
1. connect to Oracle
2. inspect or run the repo SQL scripts in order
3. verify the loaded objects
4. run UC1, UC2, and UC3
5. inspect the APEX and MCP integration points

## Repo-Aligned SQL Flow

The notebook uses the repo's SQL files as the source of truth. This keeps the operational logic versioned and easy to reuse, while the notebook remains the guided entry point.

Canonical files in order:
- `sql/01_schema.sql`
- `sql/02_staging_ddl.sql`
- `sql/03_load_onnx_model.sql`
- `sql/04_copy_data.sql`
- `sql/05_transform.sql`
- `sql/06_embed_and_index.sql`
- `sql/07_property_graph.sql`
- `sql/08_select_ai_profile.sql`
- `sql/09_uc1_card_view.sql`
- `sql/10_uc2_abandoned_app.sql`
- `sql/11_uc3_declined_txn.sql`

The next cell shows how to iterate through those files from the notebook.

In [ ]:
RUN_REPO_SQL = False
REPO_SQL_FILES = [
    '01_schema.sql',
    '02_staging_ddl.sql',
    '03_load_onnx_model.sql',
    '04_copy_data.sql',
    '05_transform.sql',
    '06_embed_and_index.sql',
    '07_property_graph.sql',
    '08_select_ai_profile.sql',
    '09_uc1_card_view.sql',
    '10_uc2_abandoned_app.sql',
    '11_uc3_declined_txn.sql',
]

for file_name in REPO_SQL_FILES:
    path = Path(SQL_ROOT) / file_name
    print(f'{file_name}: {path}')
    if RUN_REPO_SQL and path.exists():
        sql_text = path.read_text()
        print(f'Executing {file_name}...')
        run_sql(sql_text)
    elif not path.exists():
        print('  missing in this environment')
    else:
        print('  skipped')

print('Keep the SQL files in the repo; use the notebook to orchestrate them.')

## UC1, UC2, And UC3

These are the same three use cases from the repo, preserved in notebook form so the flow remains visible even when you are not opening the SQL files directly.

```sql
-- UC1: card page view + peer products + similar conversation snippets
WITH last_view AS (
  SELECT product_id
  FROM page_event
  WHERE customer_id = :cid
  ORDER BY event_ts DESC
  FETCH FIRST 1 ROW ONLY
) ,
peer_products AS (
  SELECT *
  FROM GRAPH_TABLE(
    banking_graph
    MATCH (c1 IS customer)-[:viewed]->(p IS product)<-[:viewed]-(c2 IS customer)-[:viewed]->(p2 IS product)
    WHERE c1.customer_id = :cid
      AND p.product_id = (SELECT product_id FROM last_view)
    COLUMNS (p2.product_id AS peer_product_id, p2.name AS peer_product)
  )
 )
SELECT p.peer_product,
       cc.chunk_text,
       VECTOR_DISTANCE(cc.embedding, VECTOR_EMBEDDING(MINILM_EMB USING 'credit card comparison help' AS DATA), COSINE) AS distance
FROM conversation_chunk cc
CROSS JOIN peer_products p
ORDER BY distance
FETCH FIRST 5 ROWS ONLY;

-- UC2: abandoned application + similar snippets
WITH abandoned AS (
  SELECT a.app_id, a.customer_id, a.product_id, a.updated_at, a.fields_json
  FROM application a
  WHERE a.status = 'STARTED'
    AND a.updated_at < SYSTIMESTAMP - INTERVAL '1' HOUR
)
SELECT ab.app_id,
       ab.customer_id,
       p.name AS product_name,
       cc.chunk_text,
       VECTOR_DISTANCE(cc.embedding, VECTOR_EMBEDDING(MINILM_EMB USING 'application abandoned income verification step' AS DATA), COSINE) AS distance
FROM abandoned ab
JOIN product p ON p.product_id = ab.product_id
CROSS JOIN conversation_chunk cc
ORDER BY distance
FETCH FIRST 10 ROWS ONLY;

-- UC3: declined transaction + Select AI
SELECT DBMS_CLOUD_AI.GENERATE(prompt => 'Customer 1001 just had a declined transaction because the daily limit was reached. Craft a short proactive nudge.', action => 'chat')
FROM dual;
```

In [ ]:
DROP_DEMO_OBJECTS = False

validation_sql = 'SELECT COUNT(*) AS row_count FROM customer'
print(validation_sql)

if DROP_DEMO_OBJECTS:
    for stmt in [
        'DROP PROPERTY GRAPH banking_graph',
        'DROP INDEX product_vec_idx',
        'DROP INDEX conversation_chunk_vec_idx',
        'DROP INDEX offer_vec_idx',
        'DROP TABLE offer PURGE',
        'DROP TABLE conversation_chunk PURGE',
        'DROP TABLE txn PURGE',
        'DROP TABLE application PURGE',
        'DROP TABLE page_event PURGE',
        'DROP TABLE product PURGE',
        'DROP TABLE customer PURGE',
    ]:
        try:
            run_sql(stmt)
        except Exception:
            pass
    print('Demo objects dropped.')
else:
    print('Cleanup skipped.')

## MCP And APEX

```json
{
  "mcpServers": {
    "oracle-adb": {
      "command": "sql",
      "args": ["-mcp"],
      "env": {
        "TNS_ADMIN": "/path/to/wallet"
      }
    }
  }
}
```

```sql
CREATE OR REPLACE PACKAGE nudge_chat_api AS
  FUNCTION get_nudge(p_use_case IN VARCHAR2, p_customer_id IN NUMBER) RETURN CLOB;
END nudge_chat_api;
/

CREATE OR REPLACE PACKAGE BODY nudge_chat_api AS
  FUNCTION get_nudge(p_use_case IN VARCHAR2, p_customer_id IN NUMBER) RETURN CLOB IS
    l_out CLOB;
  BEGIN
    IF p_use_case = 'UC1' THEN
      l_out := TO_CLOB('I see you viewed a card product recently. Want a quick comparison?');
    ELSIF p_use_case = 'UC2' THEN
      l_out := TO_CLOB('Looks like your application is still in progress. Need help to finish it?');
    ELSIF p_use_case = 'UC3' THEN
      SELECT DBMS_CLOUD_AI.GENERATE(prompt => 'Customer ' || p_customer_id || ' just had a declined transaction. Craft a one-sentence proactive nudge.', action => 'chat') INTO l_out FROM dual;
    ELSE
      l_out := TO_CLOB('Unsupported use case. Use UC1, UC2, or UC3.');
    END IF;
    RETURN l_out;
  END get_nudge;
END nudge_chat_api;
/
```

Use MCP when an agent needs tool access to the live database session. Use the APEX package when the front end needs a simple PL/SQL entry point.

## Summary

This notebook keeps the demo self-contained and readable without forcing you to jump between documents or SQL files.

For production code, keep reusable functions outside the notebook. For this demo, the notebook is intentionally the only artifact you need.